#  PS S6E6: Enhanced CatBoost with Astronomical Feature Engineering

This notebook improves upon a baseline CatBoost model by incorporating astronomy-inspired features and stratified cross-validation.

**OOF Accuracy:** 96.64%

## Objective

Classify celestial objects into:

- GALAXY
- QSO
- STAR

using photometric measurements, redshift information, and spectral characteristics.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier

In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

print(train.shape)
print(test.shape)


(577347, 12)
(247435, 11)


##  Feature Engineering

To improve model performance, several astronomy-inspired features were created.

### Color Indices
- u − g
- g − r
- r − i
- i − z

### Extended Color Features
- u − r
- g − i
- r − z

### Magnitude Statistics
- Mean magnitude
- Standard deviation
- Minimum magnitude
- Maximum magnitude
- Magnitude range

These features help capture spectral and photometric relationships commonly used in astronomical object classification.

In [3]:
def create_features(df):

    df = df.copy()

    # Color indices
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    # Extended color features
    df["u_r"] = df["u"] - df["r"]
    df["g_i"] = df["g"] - df["i"]
    df["r_z"] = df["r"] - df["z"]

    # Magnitude statistics
    mag_cols = ["u", "g", "r", "i", "z"]

    df["mag_mean"] = df[mag_cols].mean(axis=1)
    df["mag_std"] = df[mag_cols].std(axis=1)
    df["mag_min"] = df[mag_cols].min(axis=1)
    df["mag_max"] = df[mag_cols].max(axis=1)
    df["mag_range"] = df["mag_max"] - df["mag_min"]

    return df


train = create_features(train)
test = create_features(test)

In [4]:
# TARGET ENCODING
target_encoder = LabelEncoder()
train["class"] = target_encoder.fit_transform(train["class"])

In [5]:
# FEATURES
TARGET = "class"

FEATURES = [c for c in train.columns if c not in ["id", TARGET]]

X = train[FEATURES]
y = train[TARGET]

X_test = test[FEATURES]



# CATEGORICAL FEATURES
cat_cols = [
    "spectral_type",
    "galaxy_population"
]

cat_feature_indices = [
    X.columns.get_loc(col)
    for col in cat_cols
]

##  Model

### CatBoost Classifier

Key advantages:

- Native categorical feature handling
- Strong performance on tabular data
- Minimal preprocessing requirements
- Robust handling of feature interactions

## Validation Strategy

The model was evaluated using:

- 5-Fold Stratified Cross Validation
- Out-of-Fold (OOF) predictions

This ensures a reliable estimate of generalization performance while preserving class distributions across folds.

In [6]:
# CROSS VALIDATION

N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(train))
test_preds = np.zeros((len(test), len(np.unique(y))))

In [7]:
# TRAINING
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    print("=" * 50)
    print(f"FOLD {fold}")
    print("=" * 50)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=5000,
        depth=8,
        learning_rate=0.03,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        random_seed=42,
        l2_leaf_reg=5,
        subsample=0.8,
        bootstrap_type="Bernoulli",
        early_stopping_rounds=300,
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        cat_features=cat_feature_indices,
        use_best_model=True
    )

    valid_pred = model.predict(X_valid).astype(int).flatten()

    oof_preds[valid_idx] = valid_pred

    test_preds += (
        model.predict_proba(X_test)
        / N_SPLITS
    )



FOLD 1
0:	learn: 0.9299077	test: 0.9304495	best: 0.9304495 (0)	total: 933ms	remaining: 1h 17m 42s
200:	learn: 0.9560013	test: 0.9566554	best: 0.9566554 (200)	total: 2m 40s	remaining: 1h 3m 59s
400:	learn: 0.9614205	test: 0.9612367	best: 0.9612367 (400)	total: 5m 20s	remaining: 1h 1m 16s
600:	learn: 0.9637111	test: 0.9636962	best: 0.9636962 (600)	total: 7m 59s	remaining: 58m 26s
800:	learn: 0.9650123	test: 0.9643111	best: 0.9643197 (798)	total: 10m 35s	remaining: 55m 33s
1000:	learn: 0.9660039	test: 0.9645882	best: 0.9646228 (999)	total: 13m 11s	remaining: 52m 40s
1200:	learn: 0.9667162	test: 0.9648567	best: 0.9648740 (1197)	total: 15m 45s	remaining: 49m 50s
1400:	learn: 0.9674827	test: 0.9651165	best: 0.9651771 (1374)	total: 18m 19s	remaining: 47m 4s
1600:	learn: 0.9681127	test: 0.9653590	best: 0.9653763 (1597)	total: 20m 52s	remaining: 44m 19s
1800:	learn: 0.9687211	test: 0.9655668	best: 0.9655841 (1794)	total: 23m 31s	remaining: 41m 47s
2000:	learn: 0.9692234	test: 0.9657140	best: 0.

In [8]:
# OOF SCORE
# =====================================================

oof_score = accuracy_score(y, oof_preds)

print("\n")
print("=" * 50)
print("OOF Accuracy:", oof_score)
print("=" * 50)



OOF Accuracy: 0.9663997561258654


## Results

| Metric | Score |
|----------|----------|
| OOF Accuracy | 96.64% |

The engineered features provided a modest improvement over the baseline CatBoost model.

##  Submission

Predictions on the test dataset were averaged across all folds and exported in the competition submission format.

In [9]:
# SUBMISSION

final_preds = np.argmax(test_preds, axis=1)

final_preds = target_encoder.inverse_transform(final_preds)

submission = sample_sub.copy()

submission["class"] = final_preds

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv saved!")

submission.head()


submission.csv saved!


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


##  Future Work

Potential improvements include:

- Hyperparameter optimization
- LightGBM and XGBoost models
- Ensemble learning
- Additional astrophysical feature engineering
- Pseudo-labeling

## Conclusion

By combining CatBoost with astronomy-inspired feature engineering, the model achieved an **OOF Accuracy of 96.64%**.

This notebook provides a strong and reproducible baseline for further leaderboard improvements.